In [ ]:
!pip install -q --upgrade pip setuptools wheel
!pip install -q datasets transformers accelerate
!pip install -q seqeval --no-build-isolation

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu130 requires setuptools<82, but you have setuptools 84.0.0 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [ ]:
!pip install -q datasets transformers seqeval accelerate

In [ ]:
import unicodedata
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/indic-ner"
CKPT_DIR = f"{DRIVE_DIR}/indicbert_hi_best"

tokenizer = AutoTokenizer.from_pretrained(CKPT_DIR)
model = AutoModelForTokenClassification.from_pretrained(CKPT_DIR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device).eval()
id2label = model.config.id2label

LANGS = ["hi", "bn", "ta", "te"]

def normalize_tokens(example):
    example["tokens"] = [
        unicodedata.normalize("NFC", t) for t in example["tokens"]
    ]
    return example

test_sets = {}
for lang in LANGS:
    ds = load_dataset(
        "parquet",
        data_files={"test": f"hf://datasets/ai4bharat/naamapadam@~parquet/{lang}/test/0000.parquet"},
    )["test"]
    bad = sum(1 for ex in ds if len(ex["tokens"]) != len(ex["ner_tags"]))
    assert bad == 0, f"{lang}: {bad} misaligned rows"
    ds = ds.map(normalize_tokens, desc=f"NFC {lang}")
    test_sets[lang] = ds
    print(lang, len(ds), "gold test rows")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

hi 867 gold test rows
bn 607 gold test rows
ta 758 gold test rows
te 847 gold test rows


In [ ]:
from transformers import DataCollatorForTokenClassification, Trainer

MAX_LEN = 128

def tokenize_and_align(batch):
    enc = tokenizer(
        batch["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LEN,
    )
    aligned = []
    for i, tags in enumerate(batch["ner_tags"]):
        word_ids = enc.word_ids(batch_index=i)
        labels, prev = [], None
        for wid in word_ids:
            if wid is None:
                labels.append(-100)
            elif wid != prev:
                labels.append(tags[wid])
            else:
                labels.append(-100)
            prev = wid
        aligned.append(labels)
    enc["labels"] = aligned
    return enc

trainer = Trainer(model=model, data_collator=DataCollatorForTokenClassification(tokenizer))

predictions = {}
for lang, ds in test_sets.items():
    tok = ds.map(
        tokenize_and_align,
        batched=True,
        remove_columns=ds.column_names,
        desc=f"Tokenizing {lang}",
    )
    out = trainer.predict(tok)
    preds = np.argmax(out.predictions, axis=-1)
    y_true, y_pred = [], []
    for pred, lab in zip(preds, out.label_ids):
        keep = lab != -100
        y_true.append([id2label[l] for l in lab[keep]])
        y_pred.append([id2label[p] for p in pred[keep]])
    predictions[lang] = (y_true, y_pred)
    print(lang, "done")

hi done


bn done


ta done


te done


In [ ]:
from seqeval.metrics import classification_report, f1_score

final_f1 = {}
for lang, (y_true, y_pred) in predictions.items():
    final_f1[lang] = f1_score(y_true, y_pred)
    role = "in-language" if lang == "hi" else "zero-shot"
    print(f"===== {lang} ({role}) | gold-test entity-F1: {final_f1[lang]:.4f} =====")
    print(classification_report(y_true, y_pred))

===== hi (in-language) | gold-test entity-F1: 0.8274 =====
           precision    recall  f1-score   support

      PER       0.85      0.93      0.89       790
      LOC       0.81      0.85      0.83       614
      ORG       0.69      0.78      0.73       525

micro avg       0.80      0.86      0.83      1929
macro avg       0.80      0.86      0.83      1929

===== bn (zero-shot) | gold-test entity-F1: 0.8082 =====
           precision    recall  f1-score   support

      ORG       0.74      0.73      0.73       207
      PER       0.83      0.86      0.84       458
      LOC       0.81      0.80      0.80       331

micro avg       0.80      0.81      0.81       996
macro avg       0.80      0.81      0.81       996

===== ta (zero-shot) | gold-test entity-F1: 0.7506 =====
           precision    recall  f1-score   support

      PER       0.82      0.87      0.84       484
      ORG       0.56      0.66      0.60       304
      LOC       0.73      0.79      0.76       389

mic

In [ ]:
rng = np.random.default_rng(42)

def bootstrap_f1_ci(y_true, y_pred, n_boot: int = 1000):
    n = len(y_true)
    scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        scores.append(f1_score([y_true[i] for i in idx], [y_pred[i] for i in idx]))
    return float(np.percentile(scores, 2.5)), float(np.percentile(scores, 97.5))

print(f"{'lang':<6}{'rows':>6}{'F1':>9}{'95% CI':>22}{'role':>14}")
print("-" * 57)
ci_results = {}
for lang in LANGS:
    y_true, y_pred = predictions[lang]
    lo, hi_ = bootstrap_f1_ci(y_true, y_pred)
    ci_results[lang] = {"f1": final_f1[lang], "ci_low": lo, "ci_high": hi_, "n": len(y_true)}
    role = "in-language" if lang == "hi" else "zero-shot"
    print(f"{lang:<6}{len(y_true):>6}{final_f1[lang]:>9.4f}   [{lo:.4f}, {hi_:.4f}]{role:>14}")

lang    rows       F1                95% CI          role
---------------------------------------------------------
hi       867   0.8274   [0.8066, 0.8484]   in-language
bn       607   0.8082   [0.7792, 0.8342]     zero-shot
ta       758   0.7506   [0.7235, 0.7767]     zero-shot
te       847   0.8531   [0.8301, 0.8747]     zero-shot


In [ ]:
HI_VAL_F1 = 0.78

print(f"hi validation (silver labels): {HI_VAL_F1:.4f}")
print(f"hi gold test (manual labels):  {final_f1['hi']:.4f}")
print(f"silver-vs-gold gap:            {final_f1['hi'] - HI_VAL_F1:+.4f}")
print()
print("Interpretation for the report:")
print("If gold > silver: the model is better than silver-label eval suggested,")
print("i.e. part of the 'error' was noisy silver references, not model failure.")

hi validation (silver labels): 0.7800
hi gold test (manual labels):  0.8274
silver-vs-gold gap:            +0.0474

Interpretation for the report:
If gold > silver: the model is better than silver-label eval suggested,
i.e. part of the 'error' was noisy silver references, not model failure.


In [ ]:
import json

final = {
    "experiment": "Final gold-test evaluation (run once)",
    "model": "IndicBERTv2-MLM-only fine-tuned on 100k silver hi train",
    "baseline": {"bilstm_crf_hi_validation_f1": 0.7406},
    "gold_test": ci_results,
    "hi_validation_silver_f1": HI_VAL_F1,
    "protocol": {
        "train": "naamapadam hi train, 100k subsample, seed 42, silver projected labels",
        "model_selection": "validation entity-F1 only",
        "final_eval": "official manually annotated gold test sets, bootstrap 95% CI (1000 resamples)",
    },
}
path = f"{DRIVE_DIR}/final_gold_test_results.json"
with open(path, "w") as f:
    json.dump(final, f, indent=2)
print("Saved:", path)
print(json.dumps(final, indent=2))

Saved: /content/drive/MyDrive/indic-ner/final_gold_test_results.json
{
  "experiment": "Final gold-test evaluation (run once)",
  "model": "IndicBERTv2-MLM-only fine-tuned on 100k silver hi train",
  "baseline": {
    "bilstm_crf_hi_validation_f1": 0.7406
  },
  "gold_test": {
    "hi": {
      "f1": 0.8274490303331676,
      "ci_low": 0.8065922779351065,
      "ci_high": 0.8483677679280222,
      "n": 867
    },
    "bn": {
      "f1": 0.8081918081918081,
      "ci_low": 0.7791750699275622,
      "ci_high": 0.8342032509251719,
      "n": 607
    },
    "ta": {
      "f1": 0.7506053268765135,
      "ci_low": 0.7234550477897251,
      "ci_high": 0.7766996973907976,
      "n": 758
    },
    "te": {
      "f1": 0.8531216167448574,
      "ci_low": 0.83011629381048,
      "ci_high": 0.8746700193588642,
      "n": 847
    }
  },
  "hi_validation_silver_f1": 0.78,
  "protocol": {
    "train": "naamapadam hi train, 100k subsample, seed 42, silver projected labels",
    "model_selection": "val